# Capítulo 3: IA para la Detección de Amenazas *(Versión Simulada)*

## 3.1. Introducción

Esta versión utiliza un **dataset sintético** generado automáticamente que replica la estructura y propiedades estadísticas del tráfico de red real. Permite ejecutar y validar todo el pipeline sin necesidad de datos reales.

En este notebook implementaremos:
- **Generación** de datos simulados de tráfico de red.
- **Carga y exploración** del dataset.
- **Preprocesamiento y normalización** de las características.
- **Detección de anomalías** con Isolation Forest.
- **Detección de intrusiones** con Autoencoder (Deep Learning).

---
## 3.2. Generación de datos simulados

Se generan **5,250 registros** de tráfico de red:
- **5,000** muestras de tráfico normal (patrones estables, puertos estándar, volumen moderado).
- **250** muestras anómalas (~5%): volúmenes elevados, puertos sospechosos, duraciones inusuales.

In [ ]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)
os.makedirs('data', exist_ok=True)

n_normal = 5000
n_anomaly = 250

# Tráfico normal
normal = pd.DataFrame({
    'bytes_sent':  np.random.exponential(500, n_normal).astype(int),
    'bytes_recv':  np.random.exponential(800, n_normal).astype(int),
    'duration':    np.random.exponential(30, n_normal).round(2),
    'protocol':    np.random.choice(['TCP', 'UDP', 'ICMP'], n_normal, p=[0.7, 0.25, 0.05]),
    'src_port':    np.random.randint(1024, 65535, n_normal),
    'dst_port':    np.random.choice([80, 443, 53, 22, 8080], n_normal),
    'label':       0
})

# Tráfico anómalo
anomaly = pd.DataFrame({
    'bytes_sent':  np.random.exponential(5000, n_anomaly).astype(int),
    'bytes_recv':  np.random.exponential(50, n_anomaly).astype(int),
    'duration':    np.random.exponential(200, n_anomaly).round(2),
    'protocol':    np.random.choice(['TCP', 'UDP', 'ICMP'], n_anomaly, p=[0.3, 0.3, 0.4]),
    'src_port':    np.random.randint(1, 1024, n_anomaly),
    'dst_port':    np.random.choice([4444, 31337, 6667, 9999], n_anomaly),
    'label':       1
})

data_sim = pd.concat([normal, anomaly], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
data_sim.to_csv('data/network_traffic.csv', index=False)

print(f"[OK] Dataset generado: data/network_traffic.csv")
print(f"     Total: {len(data_sim)} | Normal: {n_normal} | Anomalía: {n_anomaly}")
print(f"\nResumen estadístico:")
data_sim[['bytes_sent','bytes_recv','duration','src_port']].describe().round(2)

---
## 3.3. Carga y exploración del dataset

### 3.3.1. Carga de datos de tráfico de red

In [ ]:
# Listing 3.1: Carga y exploración de datos de tráfico de red

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.read_csv('data/network_traffic.csv')

print("=== Información del dataset ===")
print(data.info())
print("\n=== Primeras filas ===")
print(data.head())
print("\n=== Estadísticas descriptivas ===")
print(data.describe())
print("\n=== Valores nulos por columna ===")
print(data.isnull().sum())

In [ ]:
# Visualización de distribución por clase
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(['bytes_sent', 'duration', 'src_port']):
    data[data['label']==0][col].hist(bins=50, alpha=0.6, ax=axes[i], color='steelblue', label='Normal', density=True)
    data[data['label']==1][col].hist(bins=50, alpha=0.6, ax=axes[i], color='red', label='Anomalía', density=True)
    axes[i].set_title(f'Distribución: {col}')
    axes[i].legend()
plt.tight_layout()
plt.savefig('data/exploracion_trafico.png', dpi=150)
plt.show()

### 3.3.2. Preprocesamiento y normalización

In [ ]:
# Listing 3.2: Preprocesamiento completo del dataset de tráfico

from sklearn.preprocessing import MinMaxScaler, LabelEncoder

data = pd.read_csv('data/network_traffic.csv').dropna()

le = LabelEncoder()
if data['protocol'].dtype == object:
    data['protocol'] = le.fit_transform(data['protocol'])

feature_cols = ['bytes_sent', 'bytes_recv', 'duration', 'src_port', 'dst_port', 'protocol']
features = data[feature_cols].copy()

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(features)
scaled_df = pd.DataFrame(scaled_data, columns=feature_cols)

print("Datos escalados (primeras 5 filas):")
print(scaled_df.head())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(features['bytes_sent'], bins=50, color='steelblue', alpha=0.7)
plt.title('Distribución original - bytes_sent')
plt.subplot(1, 2, 2)
plt.hist(scaled_df['bytes_sent'], bins=50, color='darkorange', alpha=0.7)
plt.title('Distribución normalizada - bytes_sent')
plt.tight_layout()
plt.savefig('data/network_distribution.png', dpi=150)
plt.show()

---
## 3.4. Detección de anomalías con Isolation Forest

In [ ]:
# Listing 3.3: Modelo de detección de anomalías con Isolation Forest

from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report

data = pd.read_csv('data/network_traffic.csv').dropna()
le = LabelEncoder()
if data['protocol'].dtype == object:
    data['protocol'] = le.fit_transform(data['protocol'])

feature_cols = ['bytes_sent', 'bytes_recv', 'duration', 'src_port']
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data[feature_cols])

model = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
model.fit(scaled_data)

predictions = model.predict(scaled_data)
scores = model.decision_function(scaled_data)
data['anomaly'] = predictions
data['anomaly_score'] = scores

anomalous = data[data['anomaly'] == -1]
normal_det = data[data['anomaly'] == 1]

print(f"Total de muestras: {len(data)}")
print(f"Muestras normales: {len(normal_det)}")
print(f"Anomalías detectadas: {len(anomalous)}")

# Evaluación contra etiquetas reales
y_pred = (predictions == -1).astype(int)
y_true = data['label']
print("\n=== Reporte de Clasificación ===")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomalía']))

plt.figure(figsize=(10, 5))
plt.scatter(range(len(normal_det)), normal_det['bytes_sent'], s=5, c='steelblue', alpha=0.4, label='Normal')
plt.scatter(anomalous.index, anomalous['bytes_sent'], s=20, c='red', alpha=0.8, label='Anomalía')
plt.xlabel('Índice de muestra')
plt.ylabel('bytes_sent')
plt.title('Anomalías detectadas en tráfico de red (Simulado)')
plt.legend()
plt.tight_layout()
plt.savefig('data/anomaly_detection.png', dpi=150)
plt.show()

---
## 3.5. Detección de intrusiones con Autoencoder (Deep Learning)

In [ ]:
# Listing 3.4: Autoencoder para detección de intrusiones

from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

data = pd.read_csv('data/network_traffic.csv').dropna()
le = LabelEncoder()
if data['protocol'].dtype == object:
    data['protocol'] = le.fit_transform(data['protocol'])

feature_cols = ['bytes_sent', 'bytes_recv', 'duration', 'src_port']
scaler_ae = MinMaxScaler()
X_all = scaler_ae.fit_transform(data[feature_cols])
labels = data['label'].values

X_normal = X_all[labels == 0]
X_train, X_val = train_test_split(X_normal, test_size=0.2, random_state=42)
X_test = X_all
y_test = labels

input_dim = X_train.shape[1]

def build_autoencoder(input_dim, encoding_dim=8):
    inp = keras.Input(shape=(input_dim,))
    encoded = layers.Dense(16, activation='relu')(inp)
    encoded = layers.Dense(encoding_dim, activation='relu')(encoded)
    decoded = layers.Dense(16, activation='relu')(encoded)
    decoded = layers.Dense(input_dim, activation='sigmoid')(decoded)
    autoencoder = keras.Model(inp, decoded, name='autoencoder')
    return autoencoder

autoencoder = build_autoencoder(input_dim)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

In [ ]:
history = autoencoder.fit(X_train, X_train, epochs=50, batch_size=64,
                          validation_data=(X_val, X_val), verbose=1)

X_test_pred = autoencoder.predict(X_test)
mse_err = np.mean(np.power(X_test - X_test_pred, 2), axis=1)

X_train_pred = autoencoder.predict(X_train)
train_mse = np.mean(np.power(X_train - X_train_pred, 2), axis=1)
threshold = np.percentile(train_mse, 95)
print(f"Umbral de detección: {threshold:.6f}")

anomalies_detected = (mse_err > threshold).astype(int)
print(f"Anomalías detectadas en test: {anomalies_detected.sum()}")

print("\n=== Reporte de Clasificación (Autoencoder) ===")
print(classification_report(y_test, anomalies_detected, target_names=['Normal', 'Anomalía']))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['loss'], label='Entrenamiento')
axes[0].plot(history.history['val_loss'], label='Validación')
axes[0].set_title('Curva de aprendizaje del Autoencoder')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].hist(mse_err[y_test == 0], bins=50, alpha=0.6, color='steelblue', label='Normal')
axes[1].hist(mse_err[y_test == 1], bins=50, alpha=0.6, color='red', label='Anomalía')
axes[1].axvline(threshold, color='black', linestyle='--', label=f'Umbral ({threshold:.4f})')
axes[1].set_title('Error de reconstrucción')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/autoencoder_loss.png', dpi=150)
plt.show()

---
## 3.6. Guardado de artefactos

In [ ]:
import joblib

os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/isolation_forest.pkl')
autoencoder.save('models/autoencoder_ids.keras')
joblib.dump(scaler, 'models/scaler_network.pkl')

data['anomaly_ae'] = anomalies_detected
data.to_csv('data/network_traffic_predictions.csv', index=False)

print("[OK] Artefactos guardados:")
print("     models/isolation_forest.pkl")
print("     models/autoencoder_ids.keras")
print("     models/scaler_network.pkl")
print("     data/network_traffic_predictions.csv")